# PyA GUI examples

- this notebook demonstrates GUIs for interacting with pya and audio data
- the first GUI integrated is Scope, a realtime oscilloscope and freqscope for
  the Aserver.
- next an Aserver dashboard (control interface) and Asig viewer will be added

## Scope

- Scope offers a simple lightweight fast and unobtrusive realtime display 
for audio data. 
- the main purpose is to use it in Aserver for r/t-monitoring, yet Scope
  can also be created independent of it for any other display purpose
- to minimize interference with time critical components (such as Aserver)
  - Scope is started via subprocess as completely independent process
  - commands are sent via pipe and processed line by line
  - shared memory is used for data transmission, i.e. it involves merely a memcopy on the user's side - all subsequent computations and gui rendering happens in the separated GUI process.


In [ ]:
from pya.gui import Scope
scp = Scope(num_samples=512, num_channels=2, pos=(-400, 0), size=(400, 300), rate=20, mode="signal")

In [ ]:
# start the scope
scp.start()

In [ ]:
# update data as needed (this is also done in Aserver if scope is used)
import numpy as np 
scp.set_data(np.random.random((512, 2))-0.5)

In [ ]:
# move window programmatically
scp.move(-400, 100)

In [ ]:
# resize GUI window programmatically
scp.resize(320, 200)

In [ ]:
# change the GUI process render/update framerate
scp.render_framerate(50)

In [ ]:
# stop the thread (if needed)
scp.stop()

In [ ]:
# exit the GUI - this closes the process
scp.exit() # alternatively use del(scp)

## Scope as Oscilloscope in Aserver

Scope is integrated into Aserver via the scope_gui() method

In [ ]:
from pya import startup
from pya.agen.lib import WhiteNoise, Line, Env

In [ ]:
s = startup(bs=1024) 
s.scope_gui()

The scope is already running. Play anything via Asig.play() or using Agen...

In [ ]:
(WhiteNoise() * Env([0.5,0,0.5, 0], [1,1, 0.2])).dup(2).play();

The Scope instance can be accessed via s.scope. All functions demonstrated above
- move, resize, render_framerate, etc. - are available.

In [ ]:
s.scope.data # this is always the current data (i.e. shared memory)

In [ ]:
from pya import Asig

Asig(s.scope.data).plot(offset=0.2, lw=0.5)

- Note that Scope only updates the view when new data are set.
- In Aserver, once all events have been processed, no further data is written
- In consequence, the last written block remains visible in the GUI.
- Also note, that you can resize the window, zoom in, pan however you like!
  - The GUI uses pyqtgraph, clicking the right button gives access to various features
  - Try Plot Options -> Transforms -> PowerSpectrum for a spectrum
  - Try Plot Options -> 

In [ ]:
from pya import Asig
s.scope.render_framerate(100)
Asig("samples/snap.wav").stereo().play(rate=0.0825)

In [ ]:
from pya.agen.lib import MouseX, MouseY, SinOsc
SinOsc(600 + MouseX(0,400) * SinOsc(300)).mul(MouseY(0.2, 0)).dup(2).play();

In [ ]:
s.stop()

In [ ]:
s.quit()

In [ ]:
s.scope.exit()

## ScopeWidget

ScopeWidget is a GUI for a scope / spectrum view in Jupyter notebooks.
- it requires matplotlib for plotting
- it requires %matplotlib widget for using interactive output cells
- it uses FuncAnimation to update the plots with data from Aserver
- ScopeWidget allows to monitor either audio input or output
  - for audio input, Aserver needs to be booted with input_flag=True
  - e.g. by using pya.startup(input_flag=True)
  - Note that for using Aserver with input, the nr. of input and output channels have to match,
  - i.e. problems arise if Input is only 1 channel (e.g. Microphone) whereas output is stereo
- notice the key bindings 'r' (reset limits), 'l' (linlog toggle) and 'a' autoscale
- fps and mode are properties, i.e. values can be set via code, the widgets will follow
- at the moment, aserver doesn't set latest_output to zeros, i.e. the latest non-zero audio output shows

In [ ]:
from pya.gui import ScopeWidget
from pya import startup

# startup Aserver with input (using the Audio Input device selected as default in your OS)
s = startup(input_flag=True)

- make sure to activate the widget mode of matplotlib.
- this requires to install ipympl

In [ ]:
# import matplotlib.pyplot as plt 
%matplotlib widget 

In [ ]:
scnb = ScopeWidget(server=Aserver.default, fps=5, mode="output")

let's play some audio so that output scope can be checked.

In [ ]:
from pya.agen.lib import BrownNoise, SinOsc
(BrownNoise() * SinOsc(1.5)).mul(0.1).dup(2).play();

set fps via the slider, or use the API, notice that it sets the widget

In [ ]:
scnb.fps = 20

let's toggle the mode (i.e. source)
- in case you have the microphone close to your laptop speaker, you should see
  a similar, properly attenuated and filtered signal

In [ ]:
scnb.mode = "input"

In [ ]:
scnb.mode = "output"


In [ ]:
s.stop() # stop any playing audio via Aserver s

User Interface:

- use Pause / Resume as needed to control the animation and background work
- use the fps slider to adjust the rendering rate in frames per second
- interactive zoom and panning is possible
- the following keybindings extend functionality:
  - 'l' for lin/log toggle: toggles through all 4 combinations of lin/log for x/y
    - please click into the panel before
  - 'a' for autoscale: to adjust x and y ranges for actually rendered data.
  - 'r' for reset limits, when mouse hovers over the axis
- 'Quit' deletes the animation object (yet the plot remains interactive) 

If you store the class instance reference manual interaction (in code) can be done,
yet the implementation will be subject to change. 

As of now, some examples are:

In [ ]:
scnb.mode = "input"  # set source
scnb.fps = 30

scnb.axspec.set_yscale('linear')
scnb.axspec.set_yscale('log')
scnb.axspec.set_ylim(1e-5, 1e2)

scnb.axsig.set_ylim(-0.01, 0.01)
scnb.line2Dsig.set_lw(3)
scnb.line2Dsig.set_color('r')

scnb.show() # create another view below this cell

## AServerGUI

AServerGUI is a UI to simplify 

- selection of audio interfaces, 
- selection of the sampling rate
- selection of the blocksize
- startup resp. reboot of the Aserver with these parameters. 

If futhermore allows 

- to start a scope GUI, and 
- play a test tone
- to stop the playback of all AGens and Asigs on the Aserver.

At this time, this is only offered via Jupyter widgets, i.e., available in Jupyter
notebooks. The plan is to add pySide based UI for regular scripting at a later time.

In [ ]:
from pya.gui import AserverGUI
asgui = AserverGUI()

In [ ]:
asgui.show() # show a copy of that UI where needed...

## AGenPlayGUI

AGenPlayGUI adds `AGen.playx()`:

- a Jupyter-widgets-based extention of the AGen.play() method
- it displays a button widget to stop playback.
- Furthermore a dropdown widget is offered to toggle the scope mode


In [ ]:
from pya.gui import AGenPlayGUI
AGenPlayGUI();

In [ ]:
from pya.agen.lib import SinOsc
SinOsc(400).mul(0.1).dup(2).playx()

## Input Controller: qwerty_keyboard_controller

- This interface uses TKInter to process keyboard actions
- it opens a Tk Window (which might not be brought to the front, so check)
- the interface blocks and needs to be closed before focus is returned to Jupyter 

In [ ]:
from pya.gui.input_devices import qwerty_keyboard_controller
def note_on(note):
    print(f"Note On: {note}", end="\r")
qwerty_keyboard_controller(playfn=note_on, transpose=48)

## Input Controller: KeyboardControllerJupyter

- This input controller uses a thread in Jupyter and direcly processes key interactions in the browser
- in vscode, note that this causes all actions in cells and vscode to be processed both by the app and the controller
- so be careful as keybindings (e.g. double d to delete a cell) could be triggered while playing
- the thread needs to be closed, otherwise it persists, so don't delete controller without doing so
- a text widget is added, focussing on that widget logs your actions and is safe
- use the Quit button to close the Keyboard Controller, otherwise use the stop() method
- repeated cell execution shows: threads may persist.

In [ ]:
from pya.gui.input_devices import KeyboardControllerJupyter

from pya.agen.lib import SinOsc, Release
from pya import startup
from pya.agen.core import asynth
import pyamapping as pam

s = startup()

# minimal example for a fully playable NoteOn+NoteOff keyboard
note_list = [0]*127

@asynth
def syn(freq=440, gate=1):
    return SinOsc(freq) * Release(gate, 1)

def note_on(note):
    note_list[note] = syn(pam.midi_to_cps(note+48), 1).play()

def note_off(note):
    note_list[note].ctrl.gate = 0 

# start controller
controller = KeyboardControllerJupyter(note_on, note_off)

In [ ]:
controller.stop()

In [ ]:
# Example callbacks
from pya.agen.lib import Release
from pya.agen.core import asynth

@asynth
def syn(rate=1, gate=1):
    ag = SinOsc(freq=rate.gen * 440) * Release(gate=gate, duration=0.5)
    # ag = PlayAsig(asig, rate=rate, loop=1) * Release(gate=gate, duration=0.5)
    return ag.mul(0.2).dup(2)

notelist = [0] * 127

def play_note(note):
    notelist[note] = syn(rate=pam.midi_to_ratio(note-30)).play()

def stop_note(note):
    notelist[note].ctrl.gate = 0